# Logistic Regression

Regresi logistik adalah model klasifikasi yang memetakan kombinasi linear fitur menjadi probabilitas kelas. Data yang dipakai memuat informasi gizi makanan bagi penderita diabetes, dengan tiga kelas konsumsi: `In Moderation`, `Less Often`, dan `More Often`.

## Tujuan

- Menyiapkan data latih dan uji.
- Melatih serta menyetel regresi logistik.
- Menafsirkan koefisien model.
- Mengevaluasi kinerja klasifikasi.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

RANDOM_STATE = 123


## Memuat data dan eksplorasi awal


In [ ]:
DATASET_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-ML241EN-SkillsNetwork/labs/datasets/food_items.csv"
)
food_data = pd.read_csv(DATASET_URL)
food_data.dtypes


In [ ]:
food_data.head(10)


In [ ]:
feature_columns = list(food_data.iloc[:, :-1].columns)
feature_columns


In [ ]:
food_data.iloc[:, :-1].describe()


Tujuh belas kolom berisi kandungan gizi setiap makanan, seperti kalori, lemak total, protein, gula, dan sodium. Kolom terakhir adalah label kelas.


In [ ]:
food_data.iloc[:, -1].value_counts(normalize=True)


In [ ]:
food_data.iloc[:, -1].value_counts().plot.bar(color=["yellow", "red", "green"])
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()


Sebaran tiga kelas tidak seimbang. Karena targetnya lebih dari dua kelas, regresi logistik yang dipakai bersifat multinomial, bukan one-vs-rest.


## Rekayasa fitur

Seluruh fitur sudah numerik, sehingga cukup diskalakan dengan `MinMaxScaler`. Label kelas disandikan dengan `LabelEncoder` menjadi nilai 0, 1, dan 2.


In [ ]:
raw_features = food_data.iloc[:, :-1]
raw_target = food_data.iloc[:, -1].squeeze()

features = MinMaxScaler().fit_transform(raw_features)
target = LabelEncoder().fit_transform(raw_target)

print(f"The range of feature inputs are within {features.min()} to {features.max()}")
np.unique(target, return_counts=True)


## Melatih model

Penalti L2 menyusutkan koefisien tanpa menghapus fitur. Solver `lbfgs` dipakai karena cocok untuk penalti L2 pada klasifikasi multikelas.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, stratify=target, random_state=RANDOM_STATE
)
print(f"Training dataset shape, X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Testing dataset shape, X_test: {X_test.shape}, y_test: {y_test.shape}")


In [ ]:
l2_model = LogisticRegression(
    random_state=RANDOM_STATE, penalty="l2", solver="lbfgs", max_iter=1000
).fit(X_train, y_train)
l2_predictions = l2_model.predict(X_test)


In [ ]:
def evaluate_metrics(actual, predicted):
    precision, recall, f_beta, _ = precision_recall_fscore_support(actual, predicted)
    return {
        "accuracy": accuracy_score(actual, predicted),
        "recall": recall,
        "precision": precision,
        "f1score": f_beta,
    }


evaluate_metrics(y_test, l2_predictions)


Penalti L1 menyusutkan sebagian koefisien menjadi nol, sehingga fitur yang saling berkorelasi bisa dihilangkan. Solver yang dipakai adalah `saga`.


In [ ]:
l1_model = LogisticRegression(
    random_state=RANDOM_STATE, penalty="l1", solver="saga", max_iter=1000
).fit(X_train, y_train)
l1_predictions = l1_model.predict(X_test)

probabilities = l1_model.predict_proba(X_test[:1, :])[0]
print("Probabilitas kelas:", probabilities)
print("Prediksi kelas:", l1_model.predict(X_test[:1, :])[0])
evaluate_metrics(y_test, l1_predictions)


Model dengan penalti L1 memberi kinerja lebih baik pada data ini karena fitur gizinya banyak yang berkorelasi.


## Confusion matrix


In [ ]:
sns.set_context("talk")
normalized_matrix = confusion_matrix(y_test, l1_predictions, normalize="true")
ConfusionMatrixDisplay(
    confusion_matrix=normalized_matrix, display_labels=l1_model.classes_
).plot()
plt.show()


## Menafsirkan koefisien

Koefisien menunjukkan arah pengaruh setiap fitur terhadap peluang suatu kelas. Nilai positif mendorong kelas tersebut, nilai negatif menekannya.


In [ ]:
def feature_coefficients(model, label_index, columns):
    pairs = {
        feature: coefficient
        for coefficient, feature in zip(model.coef_[label_index, :], columns)
        if abs(coefficient) >= 0.01
    }
    return dict(sorted(pairs.items(), key=lambda item: item[1]))


def visualize_coefficients(coefficients):
    names = list(coefficients.keys())
    values = list(coefficients.values())
    positions = np.arange(len(names))

    figure, axis = plt.subplots(figsize=(8, 8))
    axis.barh(positions, values, align="center",
              color=["r" if value <= 0 else "g" for value in values])
    axis.set_yticks(positions)
    axis.set_yticklabels(names)
    axis.invert_yaxis()
    axis.set_xlabel("Feature Coefficients")
    plt.show()


In [ ]:
visualize_coefficients(feature_coefficients(l1_model, 1, feature_columns))


Zat gizi yang kurang sehat, seperti lemak jenuh, gula, kolesterol, dan lemak total, memiliki koefisien positif tinggi untuk kelas `Less Often`.


In [ ]:
visualize_coefficients(feature_coefficients(l1_model, 2, feature_columns))


Sebaliknya, makanan dengan kalori, karbohidrat total, dan lemak total tinggi kecil kemungkinan masuk kelas `More Often`.


## Latihan: penalti elastic-net

Penalti elastic-net menggabungkan L1 dan L2. Solver `saga` mendukung penalti ini, dengan `l1_ratio` di antara 0 dan 1.


In [ ]:
elastic_model = LogisticRegression(
    random_state=RANDOM_STATE,
    penalty="elasticnet",
    l1_ratio=0.5,
    solver="saga",
    max_iter=1000,
).fit(X_train, y_train)
elastic_predictions = elastic_model.predict(X_test)
evaluate_metrics(y_test, elastic_predictions)


In [ ]:
elastic_matrix = confusion_matrix(y_test, elastic_predictions, normalize="true")
ConfusionMatrixDisplay(
    confusion_matrix=elastic_matrix, display_labels=elastic_model.classes_
).plot()
plt.show()


In [ ]:
visualize_coefficients(feature_coefficients(elastic_model, 1, feature_columns))


## Ringkasan

Penalti L1 memberi kinerja terbaik pada data ini karena menyaring fitur yang redundan. Penalti elastic-net memberi jalan tengah antara penyusutan halus dan seleksi fitur.
